# Prepare Dynamic Wflow Handoff

Collect or reuse the shared Wflow warmup baseline, then prepare and QA the Wflow discharge time series consumed by SFINCS for one Event-Catalog event.

Stage Contract
- Requires: built Wflow submodels with SFINCS handoff gauges, AORC source access, and reviewed NHDPlus/STREAM-geo river geometry.
- Produces: shared Wflow warmup forcing, Wflow event forcing, dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA, and acceptance JSON.
- Next: run `c_run_example.ipynb` after the handoff status is accepted.


In [1]:
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display

from design_events.collect_sources import collect_aorc_wflow_baseline_warmup
from sfincs_runs.config import load_runtime
from sfincs_runs.scenarios import plan_inland_coupled_example
from wflow_runs import (
    build_event_meteo_forcing,
    plan_dynamic_wflow_handoff,
    plan_wflow_streamflow_realization,
    plan_wflow_baseline_warmup_state,
    prepare_wflow_cold_instates,
    prepare_dynamic_wflow_handoff,
    validate_river_geometry_readiness,
    validate_warmup_forcing,
    validate_wflow_staticmaps_physics,
    validate_wflow_instates,
    validate_wflow_reservoir_states,
    write_wflow_reservoir_readiness,
)

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parents[1]
repo_root = location_root.parents[1]
config, paths = load_runtime(location_root / "config.yaml")
config["wflow"]["domain_set"]["review_required"] = False
print(f"location: {location_root.name}  |  root: {location_root}")


location: austin  |  root: /home/grahamhults/projects/Flood-RM/locations/austin


## 1 - Select Event and Plan Handoff


In [2]:
EVENT_ID = None  # None -> same default event as c_run_example; or e.g. "design_0369"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_inland_coupled_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)
if example_plan.event_id is None:
    raise RuntimeError("Coupled example event could not be selected; resolve upstream plan issues before dynamic handoff prep.")
if example_plan.issues:
    print("Planning issues to review:")
    for issue in example_plan.issues:
        print("  -", issue)

event_id = str(example_plan.event_id)
catalog = pd.read_csv(location_root / CATALOG_PATH)
catalog["event_id"] = catalog["event_id"].astype(str)
selected = catalog[catalog["event_id"] == event_id].iloc[0]

handoff_plan = plan_dynamic_wflow_handoff(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(handoff_plan)

streamflow_realization = plan_wflow_streamflow_realization(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(streamflow_realization)
if streamflow_realization["status"].eq("failed").any():
    raise RuntimeError("Selected event is not ready for rainfall-driven Wflow generation (missing rainfall member or amplification reference gage).")


event_id                                                            design_0369
window_start                                                2018-10-14T00:00:00
window_end                                                  2018-10-19T00:00:00
discharge_source                                                  wflow_dynamic
state_policy                                                    shared_baseline
baseline_id                                                        baseline_90d
warmup_days                                                                90.0
warmup_baseline_root          /home/grahamhults/projects/Flood-RM/locations/...
sfincs_discharge_forcing      /home/grahamhults/projects/Flood-RM/locations/...
dynamic_handoff_acceptance    /home/grahamhults/projects/Flood-RM/locations/...
acceptance_status                                                       missing
Name: dynamic_wflow_handoff_plan, dtype: object

,check,status,message
0,catalog_streamflow_member,passed,member=08151500_20181016T000000; scale=0.91032...
1,streamflow_records,passed,path=/home/grahamhults/projects/Flood-RM/locat...
2,streamflow_members,passed,member=08151500_20181016T000000


## 2 - Verify Wflow Source Geometry and Static Maps


In [3]:
river_geometry = location_root / config["collection"]["national_hydrography"]["river_geometry"]
display(validate_river_geometry_readiness(river_geometry, raise_on_error=False))

base_root = location_root / config["wflow"].get("base_model_root", "data/wflow/base")
qa_rows = []
for submodel in config["wflow"]["domain_set"].get("submodels", []):
    model_root = base_root / str(submodel["wflow_submodel_id"])
    if (model_root / "staticmaps.nc").exists():
        report = validate_wflow_staticmaps_physics(model_root, river_upa_km2=config["inland_coupling"]["discharge_forcing"].get("river_upa_km2"), raise_on_error=False)
        report.insert(0, "submodel_id", str(submodel["wflow_submodel_id"]))
        qa_rows.append(report)
staticmap_qa = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame()
display(staticmap_qa)
if not staticmap_qa.empty and staticmap_qa["status"].eq("failed").any():
    raise RuntimeError("Wflow staticmap QA must pass before dynamic SFINCS handoff.")

reservoir_readiness = write_wflow_reservoir_readiness(config, location_root, raise_on_error=False)
display(reservoir_readiness)
if not reservoir_readiness.empty and reservoir_readiness["status"].eq("failed").any():
    raise RuntimeError("Wflow reservoir QA must pass before dynamic SFINCS handoff.")


,check,status,message
0,rivwth,passed,valid=15934; unique=50
1,qbankfull,passed,valid=15934; unique=985
2,stream_geo_source,failed,source_columns=none; has_stream_geo=False


""


,submodel_id,check,status,message
0,austin_p5u,reservoir_area_id,passed,"reservoir_ids=[1, 3, 5, 6]; area_cells=12873"
1,austin_p5u,reservoir_outlet_id,passed,"outlet_ids=[1, 3, 5, 6]; outlet_cells=4; missi..."
2,austin_p5u,reservoir_initial_depth,passed,valid_cells=4; min=3.65882; max=16.9408
3,austin_p5u,meta_reservoir_mean_outflow,passed,valid_cells=4; min=0.01; max=9.09321
4,austin_p5u,reservoir_b,passed,valid_cells=4; min=0.000746997; max=0.363729
5,austin_p5u,reservoir_e,passed,valid_cells=4; min=2; max=2
6,austin_p5u,reservoir_rating_curve,passed,valid_cells=12873; min=-999; max=3
7,austin_p5u,reservoir_storage_curve,passed,valid_cells=12873; min=-999; max=1
8,austin_p5u,reservoir_outlets,passed,reservoir_ids=4; outlet_ids=4; missing_outlet_...
9,austin_p5u,reservoir_outlet_river_distance,passed,max_distance_m=0.0641813; too_far_ids=[]


## 3 - Collect Shared Warmup AORC Forcing


In [4]:
baseline_plan = plan_wflow_baseline_warmup_state(config, location_root)
display(baseline_plan)

collect_baseline_warmup = True
force_wflow_cold_instates = True
if collect_baseline_warmup:
    warmup_collection = collect_aorc_wflow_baseline_warmup(config, paths, force=False)
    display(pd.Series(warmup_collection, name="aorc_wflow_baseline_warmup"))

warmup_forcing = validate_warmup_forcing(
    config,
    location_root,
    event_id,
    reference_time=baseline_plan["baseline_reference_time"],
    raise_on_error=False,
)
display(warmup_forcing)
if warmup_forcing["status"].eq("failed").any():
    raise RuntimeError("Collect the shared AORC warmup baseline before running dynamic Wflow handoff.")

state_bootstrap = prepare_wflow_cold_instates(config, location_root, force=force_wflow_cold_instates)
display(state_bootstrap)
if not state_bootstrap.empty and state_bootstrap["status"].eq("failed").any():
    raise RuntimeError("Create native Wflow instate/instates.nc before event replay.")

instate_readiness = validate_wflow_instates(config, location_root, raise_on_error=False)
display(instate_readiness)
reservoir_state_rows = []
for row in instate_readiness.itertuples(index=False):
    model_root = Path(row.instate).parents[1]
    if Path(row.instate).exists():
        report = validate_wflow_reservoir_states(model_root, required=True, raise_on_error=False)
        report.insert(0, "submodel_id", row.submodel_id)
        reservoir_state_rows.append(report)
reservoir_state_readiness = pd.concat(reservoir_state_rows, ignore_index=True) if reservoir_state_rows else pd.DataFrame()
display(reservoir_state_readiness)
if not instate_readiness.empty and instate_readiness["status"].eq("failed").any():
    raise RuntimeError("Create or promote native Wflow instate/instates.nc from the shared warmup baseline before event replay.")
if not reservoir_state_readiness.empty and reservoir_state_readiness["status"].eq("failed").any():
    raise RuntimeError("Promote a warmup-produced Wflow instate with reservoir_water_level before event replay.")


state_policy                                                 shared_baseline
baseline_id                                                     baseline_90d
baseline_reference_time                                  2020-11-14T00:00:00
warmup_days                                                             90.0
warmup_start                                             2020-08-16T00:00:00
warmup_end                                               2020-11-13T23:00:00
cold_state_workflow        /home/grahamhults/projects/Flood-RM/locations/...
warmup_baseline_root       /home/grahamhults/projects/Flood-RM/locations/...
warmup_precip              /home/grahamhults/projects/Flood-RM/locations/...
warmup_temp_pet            /home/grahamhults/projects/Flood-RM/locations/...
state_input                                              instate/instates.nc
state_output                                           outstate/outstates.nc
base_model_root            /home/grahamhults/projects/Flood-RM/locations/...

status                                                            collected
baseline_id                                                    baseline_90d
reference_time                                          2020-11-14T00:00:00
warmup_days                                                            90.0
warmup_start                                            2020-08-16T00:00:00
warmup_end                                              2020-11-13T23:00:00
source                                                                 AORC
source_nc                 /home/grahamhults/projects/Flood-RM/locations/...
precip_nc                 /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_nc               /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_provenance       {'source_nc': '/home/grahamhults/projects/Floo...
hydromt_wflow_contract        setup_precip_forcing + setup_temp_pet_forcing
Name: aorc_wflow_baseline_warmup, dtype: object

,file,variable,status,message
0,/home/grahamhults/projects/Flood-RM/locations/...,precip,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...
1,/home/grahamhults/projects/Flood-RM/locations/...,temp,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...


2026-06-23 16:53:52,702 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-23 16:53:52,704 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-23 16:53:52,745 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-23 16:53:52,746 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-23 16:53:52,748 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-23 16:53:53,521 - hydromt.hydromt_wflow.components.states - states - WARNING - CRS not found in states data, setting 

,submodel_id,instate,status,message
0,austin_p5u,/home/grahamhults/projects/Flood-RM/locations/...,prepared,native setup_cold_states timestamp=2020-08-16T...


,submodel_id,instate,status,message
0,austin_p5u,/home/grahamhults/projects/Flood-RM/locations/...,passed,ready


,submodel_id,check,status,message
0,austin_p5u,reservoir_water_level,passed,valid_cells=4; min=3.65882; max=16.9408


## 4 - Stage Wflow Event Meteo


In [5]:
meteo_report = build_event_meteo_forcing(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    overwrite=False,
)
display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))


event_id                                                       design_0369
window_start                                           2018-10-14T00:00:00
window_end                                             2018-10-19T00:00:00
rainfall_source_nc       /home/grahamhults/projects/Flood-RM/locations/...
rainfall_scale_factor                                             1.311069
precip_path              /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_path            /home/grahamhults/projects/Flood-RM/locations/...
precip_provenance        /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_provenance      /home/grahamhults/projects/Flood-RM/locations/...
precip_written                                                       False
temp_pet_written                                                     False
Name: wflow_event_meteo_forcing, dtype: object

## 5 - Run Dynamic Wflow Handoff QA


In [6]:
run_dynamic_wflow = True

handoff_report = prepare_dynamic_wflow_handoff(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    execute=run_dynamic_wflow,
)
display(handoff_report)


2026-06-23 16:53:59,549 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-06-23 16:53:59,774 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0369/_replay_data_catalog.yml
2026-06-23 16:53:59,806 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-23 16:53:59,806 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-23 16:53:59,824 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-23 16:53:59,824 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-23 16:53:59,825 - hydromt - log - INFO

┌ Warning: 'header' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
┌ Warning: 'params' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
[ Info: Wflow version v1.0.2
[ Info: Initialize model variables for model type sbm.
┌ Info: Cyclic parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0369/austin_p5u/staticmaps.nc.
┌ Info: Forcing parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0369/austin_p5u/inmaps-event.nc.
┌ Warning: Time dimension contains _FillValue attribute, this is not in line
│ with CF conventions.
└ @ Wflow /home/grahamhults/.julia/packages/Wflow/mJ7Ug/src/io.jl:539
┌ Info: Set river_water__external_inflow_volume_flow_rate using netCDF
└ variable river

,event_id,submodel_id,window_start,window_end,update_command,resolved_update_command,hydromt_runner_status,hydromt_runner_issue,run_command,run_output_dir,status,sfincs_discharge_forcing,sfincs_discharge_written,sfincs_discharge_source,dynamic_handoff_acceptance,acceptance_status,zero_rain_control
0,design_0369,austin_p5u,2018-10-14T00:00:00,2018-10-19T00:00:00,hydromt update wflow_sbm /home/grahamhults/pro...,/home/grahamhults/projects/Flood-RM/.venv/bin/...,project_venv,,wflow_cli /home/grahamhults/projects/Flood-RM/...,/home/grahamhults/projects/Flood-RM/locations/...,completed,/home/grahamhults/projects/Flood-RM/locations/...,True,wflow_dynamic,/home/grahamhults/projects/Flood-RM/locations/...,accepted,/home/grahamhults/projects/Flood-RM/locations/...


## 6 - Acceptance Artifact


In [7]:
acceptance_path = location_root / handoff_plan["dynamic_handoff_acceptance"]
if acceptance_path.exists():
    display(pd.read_json(acceptance_path, typ="series"))
else:
    print("Dynamic handoff planned but not executed yet:", acceptance_path)


event_id                                                  design_0369
status                                                       accepted
discharge_source                                        wflow_dynamic
discharge_nc        /home/grahamhults/projects/Flood-RM/locations/...
checks              [{'check': 'event_peak', 'status': 'passed', '...
metadata            {'warmup_state_plan': {'state_policy': 'shared...
dtype: object